In [ ]:
import pandas as pd
import numpy as np

import timeit
import time
import ast
import json
import joblib
from collections import Counter

from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn import model_selection
from sklearn.multioutput import MultiOutputClassifier
from sklearn.multiclass import OneVsRestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import hamming_loss, accuracy_score, f1_score, classification_report
from sentence_transformers import SentenceTransformer
import xgboost as xgb


# Preprocessing

In [ ]:
df = pd.read_csv('../data/movies_metadata.csv')
df.shape

In [ ]:
df = df[['overview', 'genres']]
# df = df.head(500)
df.dropna(inplace=True)

# Clean up the genres column
df['genres'] = df['genres'].apply(ast.literal_eval) \
                          .apply(lambda x: [i['name'] for i in x])


In [ ]:
counter = Counter([genre for sublist in df['genres'] for genre in sublist])
counter

In [ ]:
# Filter out rare genres
GENRE_FREQ_THRESHOLD = 10
popular_genres = [genre for genre, count in counter.items() if count >= GENRE_FREQ_THRESHOLD]

df['genres'] = df['genres'].apply(lambda genres: [genre for genre in genres if genre in popular_genres])

# Drop rows with no popular genre
df['genres'] = df['genres'].apply(lambda genres: genres if len(genres) > 0 else np.nan)
df.dropna(subset=['genres'], inplace=True)

In [ ]:
# One hot encoding of genres
mlb = MultiLabelBinarizer()
df_genres = pd.DataFrame(mlb.fit_transform(df.pop('genres')),
                          columns=mlb.classes_,
                          index=df.index)

joblib.dump(mlb, '../models/mlb.joblib')
mlb.classes_

In [ ]:
df = df.join(df_genres)
df.to_csv('../data/processed_movies_metadata.csv', index=False)

# Model training

In [ ]:
df = pd.read_csv('../data/processed_movies_metadata.csv')

X = df['overview']
y = df.drop(['overview'], axis=1)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Reset indices so that SentenceTransformer encode doesn't complain
X_train.reset_index(drop=True, inplace=True)
y_train.reset_index(drop=True, inplace=True)
X_test.reset_index(drop=True, inplace=True)
y_test.reset_index(drop=True, inplace=True)

In [ ]:
def train_model(X_train, y_train, encoder, multilabeler, estimator):
    if encoder == 'TfIdf':
        encoder_model = TfidfVectorizer().fit(X_train)
    elif encoder == 'SentenceTransformer':
        encoder_model = SentenceTransformer('all-MiniLM-L6-v2')
        encoder_model.transform = encoder_model.encode
    
    X_encoded = encoder_model.transform(X_train)
    classifier_model = multilabeler(estimator=estimator)
    classifier_model.fit(X_encoded, y_train)
    return (encoder_model, classifier_model)


In [ ]:
# for encoder in ['TfIdf', 'SentenceTransformer']:
#     for multilabeler in [OneVsRestClassifier, MultiOutputClassifier]:
#         for estimator in [DecisionTreeClassifier, RandomForestClassifier, 
#                           MultinomialNB, LogisticRegression, xgb.XGBClassifier]:
#             if encoder == 'SentenceTransformer' and estimator == MultinomialNB:
#                 continue

for encoder in ['TfIdf', 'SentenceTransformer']:
    multilabeler = OneVsRestClassifier
    for estimator in [LogisticRegression, xgb.XGBClassifier]:
        start = time.time()
        print(encoder, multilabeler.__name__, estimator.__name__)
        encoder_model, classifier_model = train_model(X_train, y_train, 
                                               encoder, multilabeler, estimator())
        y_pred = classifier_model.predict(encoder_model.transform(X_test))
        report = classification_report(y_test, y_pred, output_dict=True)
        print("F1 Score: %.4f"%(report['micro avg']['f1-score']))
        print("Hamming Loss: %.4f"%(hamming_loss(y_test, y_pred)))
        stop = time.time()
        print('time taken: %.2fs' %(stop-start))
        print()


In [ ]:
joblib.dump(encoder_model, '../models/encoder_model.joblib')
joblib.dump(classifier_model, '../models/classifier_model.joblib')


In [ ]:
# for encoder in ['TfIdf', 'SentenceTransformer']:
#     for multilabeler in [OneVsRestClassifier, MultiOutputClassifier]:
#         for estimator in [DecisionTreeClassifier, RandomForestClassifier, 
#                           MultinomialNB, LogisticRegression, xgb.XGBClassifier]:
#             if encoder == 'SentenceTransformer' and estimator == MultinomialNB:
#                 continue
#             start = time.time()
#             print(encoder, multilabeler.__name__, estimator.__name__)
#             encoder_model, genre_clf = train_model(X_train, y_train, 
#                                                    encoder, multilabeler, estimator())
#             y_pred = genre_clf.predict(encoder_model.transform(X_test))

#             # Calculate metrics
# #             print("Hamming Loss: %.4f"%(hamming_loss(y_test, y_pred)))
# #             print("Accuracy Score: %.4f"%(accuracy_score(y_test, y_pred)))
#             print("F1 Score: %.4f"%(f1_score(y_test, y_pred, average='micro')))  # average can be: micro, macro, weighted, samples
# #             report = classification_report(y_test, y_pred, output_dict=True)
# #             print("Micro F1 Score: %.4f"%(report['micro avg']['f1-score']))  # average can be: micro, macro, weighted, samples
# #             print("Macro F1 Score: %.4f"%(report['macro avg']['f1-score']))  # average can be: micro, macro, weighted, samples
# #             print("Weighted F1 Score: %.4f"%(report['weighted avg']['f1-score']))  # average can be: micro, macro, weighted, samples

# #             print("Classification Report:\n", classification_report(y_test, y_pred))

#             stop = time.time()
#             duration = stop - start
#             print('time taken: %.2fs' %(duration))
#             print()
